### Silver - Ocorrências de atendimento

Problemas identificados:
- schema variável: os campos `metadata` e `customer_code` estão presentes
  em apenas parte das 270 linhas (o leitor JSON do Spark trata os campos
  ausentes como null automaticamente)
- `event_type` com `Delay`/`delay`/`refund`/`troca`/`cancel_request`/
  `complaint`/null
- `status` com `Open`/`open`/`closed`/null
- `severity` com `High`/`high`/`medium`/`low`/null
- `created_at` em 3 formatos (try_to_timestamp utilizado pelo mesmo motivo
  descrito no notebook de vendedores, relacionado ao modo ANSI do runtime)

O campo `customer_code`, presente em apenas 9 dos 270 tickets, não é
utilizado para vincular o ticket ao cliente. O vínculo é feito via
`order_id`, que referencia o pedido associado com cobertura de 100% dos
registros.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F

df_ocorrencias_bronze = spark.table(f"{schema_name}.bronze_ocorrencias")

In [ ]:
df_ocorrencias_step1 = (
    df_ocorrencias_bronze
    .withColumn("order_id", F.upper(F.trim(F.col("order_id"))))
    .withColumn(
        "event_type_norm",
        F.when(F.col("event_type").isNull(), "Não informado").otherwise(F.initcap(F.lower(F.trim(F.col("event_type")))))
    )
    .withColumn(
        "status_norm",
        F.when(F.lower(F.trim(F.col("status"))) == "open", "Aberto")
         .when(F.lower(F.trim(F.col("status"))) == "closed", "Fechado")
         .otherwise("Não informado")
    )
    .withColumn(
        "severity_norm",
        F.when(F.lower(F.trim(F.col("severity"))) == "high", "Alta")
         .when(F.lower(F.trim(F.col("severity"))) == "medium", "Média")
         .when(F.lower(F.trim(F.col("severity"))) == "low", "Baixa")
         .otherwise("Não informado")
    )
)

In [ ]:
df_ocorrencias_step2 = df_ocorrencias_step1.withColumn(
    "created_at_parsed",
    F.coalesce(
        F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(created_at, 'yyyy/MM/dd')"),
        F.expr("try_to_timestamp(created_at, 'dd/MM/yyyy HH:mm')"),
    )
)

In [ ]:
tem_metadata = "metadata" in df_ocorrencias_step2.columns
tem_customer_code = "customer_code" in df_ocorrencias_step2.columns

df_ocorrencias_step3 = df_ocorrencias_step2
if tem_metadata:
    df_ocorrencias_step3 = (
        df_ocorrencias_step3
        .withColumn("canal_atendimento", F.col("metadata.channel"))
        .withColumn("sla_horas", F.col("metadata.sla_hours"))
    )
else:
    df_ocorrencias_step3 = (
        df_ocorrencias_step3
        .withColumn("canal_atendimento", F.lit(None).cast("string"))
        .withColumn("sla_horas", F.lit(None).cast("long"))
    )

cols = [
    F.col("ticket_id"), F.col("order_id"),
    F.col("event_type_norm").alias("event_type"),
    F.col("created_at_parsed").alias("created_at"),
    F.col("severity_norm").alias("severity"),
    F.col("status_norm").alias("status"),
    F.col("canal_atendimento"), F.col("sla_horas"),
]
if tem_customer_code:
    cols.append(F.upper(F.trim(F.col("customer_code"))).alias("customer_code_informado"))
else:
    cols.append(F.lit(None).cast("string").alias("customer_code_informado"))

df_ocorrencias_final = df_ocorrencias_step3.select(*cols)

(
    df_ocorrencias_final.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_ocorrencias")
)

print("Bronze:", df_ocorrencias_bronze.count(), "-> Silver:", df_ocorrencias_final.count())

In [ ]:
print("Sem event_type informado:", df_ocorrencias_final.filter(F.col("event_type") == "Não informado").count())
print("Sem status informado:", df_ocorrencias_final.filter(F.col("status") == "Não informado").count())
print("Com metadata preenchido:", df_ocorrencias_final.filter(F.col("canal_atendimento").isNotNull()).count(), "de", df_ocorrencias_final.count())